# EasyRAG Context assembly and packing

## Chapter position

Selection decides which citations survive. Packing decides what those citations look like when they become prompt text.

## Learning question

How does EasyRAG turn a selected citation list into a context block and a full answer prompt?

## Success criteria

- You can explain the format contract that `build_context_block()` creates.
- You can separate instruction text from evidence text inside the generated prompt.
- You can explain why `generate_answer()` is mostly orchestration rather than hidden model logic.

## Source code anchors

- `easyrag.rag.generation.packing.build_context_block`
- `easyrag.rag.generation.prompting.build_generation_prompt`
- `easyrag.rag.generation.pipeline.generate_answer`
- `easyrag.rag.types.AnswerParam`

## Method principles

- `easyrag.rag.generation.packing.build_context_block`: This helper serializes selected citations into the numbered context block that the prompt builder reuses. It is the visible bridge between citation objects and prompt text.
- `easyrag.rag.generation.prompting.build_generation_prompt`: This prompt builder combines answer instructions with the packed evidence block. It is where style, citation requirements, and abstention rules become model-facing text.
- `easyrag.rag.generation.pipeline.generate_answer`: This is the answer orchestration wrapper. It reruns citation selection, packs context, builds the prompt, calls synthesis, and returns a structured `AnswerResult`.
- `easyrag.rag.types.AnswerParam`: This dataclass is the answer-side policy bundle. It controls citation budget, context budget, answer style, and abstention rules without changing retrieval behavior.


## How the code fits together

Once selection is done, `build_context_block()` turns each citation into a numbered evidence block. `build_generation_prompt()` adds instruction lines above that block, but it does not rewrite the evidence itself. `generate_answer()` reuses the same helpers and then calls `synthesize_answer()` with the resulting prompt. That means you can debug the generation pipeline one artifact at a time: selected citations first, then packed context, then the prompt, then the answer.

## What this cell is proving

We are going to inspect the exact artifacts the generation code builds. The useful thing here is not the final answer. It is the ability to point at the packed context and say, "this line became that prompt section."

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "notebooks" / "_utils.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

_notebook_utils = importlib.import_module("notebooks._utils")
_rag = importlib.import_module("easyrag.rag")
_packing = importlib.import_module("easyrag.rag.generation.packing")
_pipeline = importlib.import_module("easyrag.rag.generation.pipeline")
_prompting = importlib.import_module("easyrag.rag.generation.prompting")
_selection = importlib.import_module("easyrag.rag.generation.selection")
_async_utils = importlib.import_module("easyrag.support.async_utils")

ensure_repo_root_on_path = _notebook_utils.ensure_repo_root_on_path
_print_json = _notebook_utils.print_json
can_use_openai_compatible_models = _notebook_utils.provider_ready
skip_message = _notebook_utils.skip_message
_stub_embedding = _notebook_utils.stub_embedding
_stub_query_model = _notebook_utils.stub_query_model
AnswerParam = _rag.AnswerParam
EasyRAG = _rag.EasyRAG
QueryParam = _rag.QueryParam
build_context_block = _packing.build_context_block
generate_answer = _pipeline.generate_answer
build_generation_prompt = _prompting.build_generation_prompt
select_answer_citations = _selection.select_answer_citations
run_sync = _async_utils.run_sync

REPO_ROOT = ensure_repo_root_on_path()

## What this cell is proving

The context block is not free-form prose. It is a numbered serialization of citations, and those numbers are the same numbers the prompt and final answer use for citation markers.

In [ ]:
packing_tmp = tempfile.TemporaryDirectory()
packing_rag = EasyRAG(
    working_dir=packing_tmp.name,
    workspace="packing-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(packing_rag.initialize_storages())
run_sync(
    packing_rag.ainsert(
        [
            "# Retrieval\nEasyRAG uses grounded retrieval and query rewrite to keep answers traceable.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis and storage review.\n",
            "# Policy\nCitation-aware answers expose the evidence budget directly to readers.\n",
        ],
        ids=["doc::retrieval", "doc::storage", "doc::policy"],
        file_paths=["docs/retrieval.md", "docs/storage.md", "docs/policy.md"],
    )
)
packing_query = "How does EasyRAG keep answers grounded?"
packing_result = run_sync(
    packing_rag.aquery(packing_query, QueryParam(mode="hybrid", chunk_top_k=5))
)
packing_param = AnswerParam(
    max_citations=2, max_context_chars=180, style="citation_aware"
)
selected_citations = select_answer_citations(
    packing_result.citations,
    max_items=packing_param.max_citations,
    max_chars=packing_param.max_context_chars,
)
context_block = build_context_block(selected_citations)

_print_json(
    {
        "selected_citations": selected_citations,
        "context_block": context_block,
    }
)

## Why this output looks like this

Each block has the same shape: marker, title, location, then a normalized snippet. The numbering comes from selection order, not from retrieval scores directly. If citation `[2]` looks odd later, start here and check whether the selected order itself already looked odd.

## What this cell is proving

The prompt has two layers you can separate mechanically. One layer is instructions. The other layer is the context block you just built. If those two layers blur together, prompt debugging gets messy fast.

In [ ]:
prompt = build_generation_prompt(packing_query, context_block, packing_param)
prompt_lines = prompt.splitlines()
evidence_start = prompt_lines.index("Retrieved evidence:")
instruction_lines = prompt_lines[:evidence_start]
evidence_lines = prompt_lines[evidence_start + 1 :]

print("Instruction layer")
print("-" * 40)
print("\n".join(instruction_lines))
print("\nEvidence layer")
print("-" * 40)
print("\n".join(evidence_lines))

## Why this output looks like this

`build_generation_prompt()` prepends the instruction lines and then drops in the context block almost verbatim. Changing `AnswerParam` affects the instruction layer first. The evidence layer still comes from `build_context_block()`.

## What this cell is proving

`generate_answer()` is mostly orchestration. If we compare the manual artifacts with the returned `AnswerResult`, they should line up exactly.

In [ ]:
packed_answer = generate_answer(
    packing_query,
    packing_result,
    answer_param=packing_param,
    answer_model_func=packing_rag.answer_model_func,
)

_print_json(
    {
        "prompt_matches_manual": packed_answer.prompt == prompt,
        "context_matches_manual": packed_answer.context_block == context_block,
        "answer": packed_answer.answer,
        "metadata": {
            "raw_citation_count": packed_answer.metadata["raw_citation_count"],
            "selected_citation_count": packed_answer.metadata[
                "selected_citation_count"
            ],
            "context_chars": packed_answer.metadata["context_chars"],
            "answer_model_used": packed_answer.metadata["answer_model_used"],
            "fallback_used": packed_answer.metadata["fallback_used"],
        },
    }
)

## Why this output looks like this

The prompt and context block should match because `generate_answer()` reruns the same helper chain you just called manually. In this deterministic notebook, `answer_model_func` is `None`, so the final sentence selection comes from the built-in fallback synthesis path. That is why `fallback_used` is `true`.

## What this cell is proving

The provider-backed path should still expose the same packed artifacts when you need them. Model-backed generation changes the answer text, but it should not make the context or prompt boundaries disappear.

In [ ]:
if not can_use_openai_compatible_models():
    print(skip_message("provider"))
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-packing-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval feeds answer packing.\n",
                    "# Policy\nCitations keep the answer inspectable.\n",
                ],
                ids=["doc::retrieval", "doc::policy"],
                file_paths=["docs/retrieval.md", "docs/policy.md"],
            )
        )
        provider_answer = run_sync(
            provider_rag.aanswer(
                "How does EasyRAG keep answers inspectable?",
                QueryParam(mode="hybrid", chunk_top_k=3),
                AnswerParam(max_citations=2, max_context_chars=160),
            )
        )
        _print_json(
            {
                "context_block": provider_answer.context_block,
                "prompt_preview": provider_answer.prompt.splitlines()[:10],
            }
        )
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- If the answer markers look wrong, inspect the selected citation order before you touch the prompt builder.
- If the prompt reads strangely, separate instruction lines from evidence lines first. Mixing those two layers makes debugging harder than it needs to be.
- Do not change answer instructions and expect citation selection to change. Selection happens earlier.

In [ ]:
run_sync(packing_rag.finalize_storages())
packing_tmp.cleanup()
print("Cleaned up the packing workspace.")

## Takeaway

Packing is the last readable step before synthesis. Once the selected citations become a numbered context block, the prompt boundary is easy to inspect and `generate_answer()` becomes much less mysterious.

Continue with [05_04_prompting_and_answer_style.ipynb](05_04_prompting_and_answer_style.ipynb) to see how answer style changes the instruction layer without changing the evidence block.